# TP - Company Org Chart  (Tree + JSON + Dict)

**Type:** practical exercise (not a LeetCode problem) &nbsp;|&nbsp; **Theme:** data structures

A company is a **tree**. The CEO is the *root*. Every person has a `name`, a
`title`, a `salary`, and a list of `reports` (the people directly under them).
A person with an empty `reports` list is a *leaf* (they manage nobody).

The whole company is stored in **`../data/company.json`**. Your job is to load
it and answer questions about it.

The point of this TP is not just to get the answers - it is to **solve the same
question in several ways** and feel the difference between them:

- **recursion** (a function that calls itself on each child),
- **iteration with a stack** (depth-first, no recursion),
- **iteration with a queue** (breadth-first, level by level),
- **a dict index** (`name -> person`) so you can look somebody up instantly
  instead of searching the tree every time.

Everything you need, you already met this week: the tree shape (your BST),
recursion (Max Depth), and the dict (Two Sum).

---

### The org chart you are working with

```
                         Amine (CEO)
                        /            \
                 Sara (CTO)         Hassan (CFO)
                /         \               \
        Youssef          Lina            Fatima
        /     \         /     \             \
    Nadia    Omar    Karim   Salma        Rachid
```


## Step 0 - Load the JSON

`json.load` turns the file into nested Python **dicts and lists**. Run this and
look at the structure: `company['name']`, `company['reports']`, and each item in
`reports` is itself a person with the same shape.

In [12]:

from collections import deque

class Person:
    def __init__(self ,id:int, name:str , title:str , salary:int):
        self.id = id
        self.name = name
        self.title = title
        self.salary = salary
class Node:
    def __init__(self,person:Person,next_person = None ,person_reports = None) :
        self.person = person
        self.next_person = next_person
        self.person_reports = LinkedList(reportOwner=person.id)
    def printList(self):
        current = self
        while current:
            print(f"employee id : {current.person.id} [ name : {current.person.name}  salary : {current.person.salary}  , title : {current.person.title} ] ")
            current = current.next_person

class LinkedList:
    def __init__(self , head:Node=None , reportOwner:int = 0) :
        self.head = head
        self.reportOwner:int = reportOwner
        self.size = 0
    def append(self,employee:Node):
        employee.next_person = self.head
        self.head = employee
        self.size +=1
    def getById(self,id:int = 0) -> Node:
        current = self.head
        while current:
            if current.person.id == id:
                return current
            current = current.next_person
        return current


class Tree:
    def __init__(self):
        self.linkedList:LinkedList = LinkedList()

    def getListByOwnerId(self,reportOwner:int)-> LinkedList:
        stack = deque()
        stack.append(self.linkedList)
        while stack:
            size = len(stack)
            for _ in range(size):
                current:LinkedList = stack.popleft()
                if current.reportOwner == reportOwner :
                    return current
                p:Node = current.head
                while p:
                    if p.person_reports:
                        stack.append(p.person_reports)
                    p = p.next_person
        return None

    def append(self,employee:Person , ChefId:int):
        if employee.id == ChefId: return
        if self.linkedList.head is None:
            self.linkedList.append(Node(employee))
            self.linkedList.reportOwner = 0      # <-- ONLY CHANGE  (was 1)
            return
        DB = self.getListByOwnerId(ChefId)
        if DB:
            DB.append(Node(employee))

def show(lst:LinkedList, indent:int=0):
    p = lst.head
    while p:
        print("   " * indent + f"- {p.person.name} (id {p.person.id})")
        show(p.person_reports, indent + 1)
        p = p.next_person

print("root list reportOwner =", tree.linkedList.reportOwner, " size =", tree.linkedList.size)
show(tree.linkedList)




### Fix cell (mine, not yours) - `TreeFixed`

Your cell above stays exactly as you wrote it. The cell below is a **copy of
`Tree` only** (`Person`, `Node`, `LinkedList` are reused from your cell, not
rewritten) with **one line changed**, marked `# <-- ONLY CHANGE`.

The diff is literally:

```python
self.linkedList.reportOwner = 1     # yours
self.linkedList.reportOwner = 0     # fixed
```

Because the rule you stated is: *a list's `reportOwner` is the id of the person
whose reports it holds*. The root list does not hold anybody's reports - it
holds the CEO himself. So it belongs to **nobody**, and `0` is your "no boss"
id (the same `0` your `printPersons` shows next to Amine).

Writing `1` made the root list claim to be *Amine's reports list* - but Amine's
`Node` already has one with `reportOwner == 1`. Two lists, same id. BFS reaches
the root first, so every `ChefId=1` append landed **next to** Amine instead of
**under** him.

Note the order in the output: Hassan, Sara, Mohamed - reversed. That is **not**
a bug, that is your `append` doing `addFirst` (O(1), the trick from Zigzag).

In [16]:



# --- just a harness so you can SEE it, nothing clever here ---
# (employee, ChefId) pairs = exactly what your printPersons already printed
tree = TreeFixed()
tree.append(Person(1,  "Amine",   "CEO",               120000), 0)
tree.append(Person(2,  "Mohamed", "Designer",           10000), 1)
tree.append(Person(3,  "Sara",    "CTO",                95000), 1)
tree.append(Person(4,  "Youssef", "Backend Lead",       70000), 3)
tree.append(Person(5,  "Nadia",   "Backend Dev",        52000), 4)
tree.append(Person(6,  "Omar",    "Backend Dev",        50000), 4)
tree.append(Person(7,  "Lina",    "Frontend Lead",      68000), 3)
tree.append(Person(8,  "Karim",   "Frontend Dev",       48000), 7)
tree.append(Person(9,  "Salma",   "Frontend Dev",       47000), 7)
tree.append(Person(10, "Hassan",  "CFO",                90000), 1)
tree.append(Person(11, "Fatima",  "Accountant",         55000), 10)
tree.append(Person(12, "Rachid",  "Junior Accountant",  38000), 11)




root list reportOwner = 0  size = 1
- Amine (id 1)
   - Hassan (id 10)
      - Fatima (id 11)
         - Rachid (id 12)
   - Sara (id 3)
      - Lina (id 7)
         - Salma (id 9)
         - Karim (id 8)
      - Youssef (id 4)
         - Omar (id 6)
         - Nadia (id 5)
   - Mohamed (id 2)


In [15]:
import json


# this notebook lives in Problems/tp/ , the data lives in Problems/data/


class Company:
    def __init__(self):
        with open("../data/company.json", encoding="utf-8") as f:
                self.company = json.load(f)

def printPersons(person):
    print(
        f"id={person['id']}, "
        f"name={person['name']}, "
        f"title={person['title']}"
    )

    for employee in person["reports"]:
        printPersons(employee)

def printPersons(person, ownerId=0):

    print(
        f"id={person['id']}   "
        f"name={person['name']}   "
        f"team owner={ownerId}"
    )

    for employee in person["reports"]:
        printPersons(employee, person["id"])

printPersons(Company().company)




id=1   name=Amine   team owner=0
id=2   name=Mohamed   team owner=1
id=3   name=Sara   team owner=1
id=4   name=Youssef   team owner=3
id=5   name=Nadia   team owner=4
id=6   name=Omar   team owner=4
id=7   name=Lina   team owner=3
id=8   name=Karim   team owner=7
id=9   name=Salma   team owner=7
id=10   name=Hassan   team owner=1
id=11   name=Fatima   team owner=10
id=12   name=Rachid   team owner=11


## Exercise 1 - Count everybody  `O(n)`

How many people work at the company (including the CEO)? **Answer: 10.**

Do it **three ways** - they should all print 10:
- `countRecursive(person)` : 1 + the count of each report (function calls itself)
- `countStack(root)` : push the root on a list, pop, push its reports, repeat
- `countQueue(root)` : same but with a queue (`collections.deque`, popleft)

In [ ]:
def countRecursive(person: dict) -> int:
    # TODO : 1 for this person, plus countRecursive of every report
    pass

def countStack(root: dict) -> int:
    # TODO : depth-first with a list used as a stack (append / pop)
    pass

def countQueue(root: dict) -> int:
    # TODO : breadth-first with a deque (append / popleft)
    pass

# print(countRecursive(company))   # 10
# print(countStack(company))       # 10
# print(countQueue(company))       # 10


## Exercise 2 - Depth of the hierarchy  `O(n)`

How many management levels are there? This is **exactly Max Depth (#104)**, just
with many children instead of only left/right. **Answer: 4** (Amine -> Sara ->
Youssef -> Nadia).

`depth(person)` = 1 + the **max** depth among its reports (a leaf has depth 1).

In [ ]:
def depth(person: dict) -> int:
    # TODO
    pass

# print(depth(company))    # 4


## Exercise 3 - Total payroll  `O(n)`

Sum of everybody's salary. **Answer: 730000.**

`totalSalary(person)` = this person's salary + the total of each report.

In [ ]:
def totalSalary(person: dict) -> int:
    # TODO
    pass

# print(totalSalary(company))    # 730000


## Exercise 4 - Find a person by name (two ways)

**Way A - search the tree** every time: `findInTree(root, name)` walks the tree
and returns the person dict (or `None`). Cost per lookup: `O(n)`.

**Way B - build a dict index once**: `buildIndex(root)` returns
`{ name: person }`. After that, `index[name]` is `O(1)` - the same lesson as
Two Sum. If you look people up many times, B wins big.

Test both with `"Karim"` (exists) and `"Ghost"` (does not).

In [ ]:
def findInTree(root: dict, name: str):
    # TODO : return the person dict whose 'name' == name, else None
    pass

def buildIndex(root: dict) -> dict:
    # TODO : return { name: person } for everybody in the tree
    pass

# print(findInTree(company, "Karim"))    # the Karim dict
# print(findInTree(company, "Ghost"))    # None
# index = buildIndex(company)
# print(index["Karim"]["salary"])        # 48000


## Exercise 5 - Cost of a team  `O(n)`

Given a manager's name, what is the total salary of **their whole team**
(themselves + everyone below them)?

Hint: you already have the tools. Find the manager (Exercise 4), then run
`totalSalary` on that sub-tree (Exercise 3). Reusing what you built is the whole
point.

- `teamCost(company, "Sara")`  ->  **430000**
- `teamCost(company, "Youssef")` -> **172000**

In [ ]:
def teamCost(root: dict, manager_name: str) -> int:
    # TODO : find the manager, then total the salary of their sub-tree
    pass

# print(teamCost(company, "Sara"))       # 430000
# print(teamCost(company, "Youssef"))    # 172000


## Exercise 6 (bonus) - Chain of command

Build a flat dict `boss = { name: manager_name }` (the CEO's boss is `None`).
Then, given a name, walk *up* that dict to print the path to the CEO.

- `chain("Nadia")`  ->  `['Nadia', 'Youssef', 'Sara', 'Amine']`

This is a tree walked **upwards** using parent pointers stored in a dict -
a different way to represent the same tree.

In [ ]:
def buildBossMap(root: dict, boss=None, table=None) -> dict:
    # TODO : fill table[person_name] = boss_name for the whole tree
    pass

def chain(name: str, boss_map: dict) -> list:
    # TODO : follow boss_map from name up to the CEO
    pass

# bosses = buildBossMap(company)
# print(chain("Nadia", bosses))    # ['Nadia', 'Youssef', 'Sara', 'Amine']


## Exercise 7 (bonus) - JSON  ->  your own node class

So far you used raw dicts. Now parse the JSON into real objects, like the
`TreeNode` you built for Max Depth:

```python
class Employee:
    def __init__(self, name, title, salary):
        self.name = name
        self.title = title
        self.salary = salary
        self.reports = []      # list of Employee
```

Write `parse(person_dict) -> Employee` (recursive) that turns the nested dict
into a tree of `Employee` objects, then redo Exercise 1 (count) on the object
tree. Same algorithm, `person.reports` instead of `person["reports"]`.

In [ ]:
class Employee:
    def __init__(self, name, title, salary):
        self.name = name
        self.title = title
        self.salary = salary
        self.reports = []

def parse(person: dict) -> "Employee":
    # TODO : build an Employee, then parse each report and append it
    pass

# ceo = parse(company)
# print(ceo.name, ceo.title)
